# 11 - Perceptrons et descente de gradient, pas à pas, sur `day.csv`

## Le problème

Le fichier `day.csv` contient **731 jours** (2011-2012) d'un service de vélos en libre-service à Washington.
Pour chaque jour on connaît la météo, la saison, si c'est un jour travaillé… et le nombre de vélos loués : `cnt`.

**Objectif : prédire `cnt` à partir des autres colonnes.** C'est un problème de **régression**
(on prédit un nombre, pas une catégorie).

## Pourquoi tout coder à la main ?

Keras fait tout en une ligne (`model.fit(...)`), mais on ne voit rien de ce qui se passe.
Ici on écrit nous-mêmes, avec numpy, chaque étape : prédiction, calcul de l'erreur, gradient, mise à jour
des poids. Une fois qu'on a compris ça, Keras n'a plus de secret.

## Petit lexique (à relire si un mot bloque)

| Mot | Signification |
|---|---|
| **neurone** | calcule une somme pondérée des entrées : $w_1 x_1 + w_2 x_2 + \dots + b$, puis applique (ou pas) une fonction |
| **poids** $w$, **biais** $b$ | les nombres que le modèle **apprend** |
| **perte** (loss) | un nombre qui mesure à quel point le modèle se trompe. On veut la rendre la plus petite possible |
| **gradient** | la « pente » de la perte par rapport à chaque poids : il indique dans quel sens bouger le poids pour que la perte augmente. On va donc dans le sens **opposé** |
| **learning rate** $\eta$ | la taille du pas qu'on fait à chaque mise à jour |
| **époque** | un passage complet sur toutes les données d'entraînement |
| **batch** | le paquet d'exemples utilisé pour calculer **une** mise à jour |
| **train / validation / test** | données pour apprendre / pour choisir les réglages / pour l'évaluation finale (on n'y touche qu'à la fin) |
| **sur-apprentissage** | le modèle apprend le train « par cœur » (y compris le bruit) et se trompe sur des données nouvelles |

## Plan

1. Charger et préparer les données
2. Baseline : la prédiction la plus bête possible
3. Un neurone, une variable : la descente de gradient à la main
4. L'effet du learning rate
5. Un neurone, toutes les variables (perceptron linéaire)
6. Batch, mini-batch ou SGD ?
7. Le perceptron historique de Rosenblatt (classification)
8. Un réseau à une couche cachée (MLP)
9. Combien de neurones cachés ?
10. Les optimiseurs : SGD, momentum, Adam
11. Lutter contre le sur-apprentissage : L2 et early stopping
12. Bilan et évaluation finale

In [ ]:
import csv                       # lire le fichier CSV (pandas n'est pas nécessaire ici)
import numpy as np               # calcul sur des tableaux de nombres
import matplotlib.pyplot as plt  # graphiques

np.random.seed(0)                # fixe le hasard : on obtient les mêmes résultats à chaque exécution
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

---
## 1. Charger et préparer les données

### 1.1 Lecture du fichier

In [ ]:
with open("../data/day.csv") as f:
    rows = list(csv.DictReader(f))   # une liste de dictionnaires : une ligne = un jour

print(len(rows), "jours")
print("Exemple de ligne :", rows[0])

Que signifient les colonnes ?

| Colonne | Contenu | On la garde ? |
|---|---|---|
| `instant`, `dteday` | numéro de ligne, date | non : ce sont des identifiants |
| `season` | 1 = printemps, 2 = été, 3 = automne, 4 = hiver | oui (en one-hot, voir plus bas) |
| `yr` | 0 = 2011, 1 = 2012 | oui (le service est plus populaire en 2012) |
| `mnth`, `weekday` | mois, jour de la semaine | non, pour rester simple (la saison et `workingday` résument l'essentiel) |
| `holiday`, `workingday` | jour férié, jour travaillé (0/1) | oui |
| `weathersit` | 1 = beau, 2 = brume, 3 = pluie/neige | oui (en one-hot) |
| `temp`, `atemp`, `hum`, `windspeed` | température, ressentie, humidité, vent (déjà ramenés entre 0 et 1) | oui |
| `casual`, `registered` | locations des occasionnels et des abonnés | **non !** |
| `cnt` | **la cible** | c'est ce qu'on prédit |

**Pourquoi enlever `casual` et `registered` ?** Parce que `casual + registered = cnt`. Si on les donne au
modèle, il n'a qu'à faire une addition : il aurait l'air parfait mais serait inutilisable en vrai (le jour même,
on ne connaît pas encore ces nombres). C'est une **fuite de données**.

### 1.2 Construire la matrice X (entrées) et le vecteur y (cible)

**Le one-hot.** `season` vaut 1, 2, 3 ou 4. Si on donnait ce nombre tel quel, le modèle croirait que
« hiver (4) = 4 × printemps (1) », ce qui n'a pas de sens. On crée donc une colonne par saison, qui vaut 1
si c'est la bonne saison et 0 sinon. Même chose pour la météo.

In [ ]:
# Variables numériques, gardées telles quelles
num_cols = ["yr", "holiday", "workingday", "temp", "atemp", "hum", "windspeed"]

# Variables catégorielles -> one-hot (ex. été = [0, 1, 0, 0])
season = [[float(int(r["season"]) == s) for s in [1, 2, 3, 4]] for r in rows]
meteo  = [[float(int(r["weathersit"]) == m) for m in [1, 2, 3]] for r in rows]

X = np.array([[float(r[c]) for c in num_cols] for r in rows])
X = np.hstack([X, season, meteo])                 # on colle toutes les colonnes côte à côte
y = np.array([float(r["cnt"]) for r in rows])     # la cible

col_names = num_cols + ["printemps", "été", "automne", "hiver"] + ["meteo_beau", "meteo_brume", "meteo_pluie"]
print("X :", X.shape, "-> 731 jours, 14 variables")
print("y :", y.shape)
print("cnt : min", y.min(), "| moyenne", round(y.mean()), "| max", y.max())

### 1.3 Un premier coup d'œil

Avant tout modèle, on regarde les données. La température est sans doute la variable la plus importante :

In [ ]:
plt.scatter(X[:, 3], y, s=8, alpha=0.6)
plt.xlabel("temp (entre 0 et 1)"); plt.ylabel("cnt (vélos loués)")
plt.title("Locations selon la température")
plt.show()

On voit une tendance claire : **plus il fait chaud, plus on loue de vélos** (avec une légère baisse quand il fait
très chaud). Mais le nuage est épais : la température seule ne suffira pas à tout expliquer.

### 1.4 Découper en train / validation / test

* **train (60 %)** : le modèle apprend dessus ;
* **validation (20 %)** : on compare nos différents essais dessus, pour choisir les réglages ;
* **test (20 %)** : on ne le regarde **qu'une seule fois, à la toute fin**, pour avoir une note honnête.

Pourquoi trois et pas deux ? Si on choisit les réglages en regardant le test, on finit par « tricher »
indirectement : la note finale serait trop optimiste.

In [ ]:
idx = np.random.permutation(len(X))              # mélange aléatoire des 731 indices
i_train, i_val, i_test = idx[:439], idx[439:585], idx[585:]
print(len(i_train), "jours de train,", len(i_val), "de validation,", len(i_test), "de test")

### 1.5 Normaliser

Les variables n'ont pas la même échelle (`yr` vaut 0 ou 1, `hum` entre 0 et 1, etc.). On les **centre-réduit** :
$$x_{\text{norm}} = \frac{x - \text{moyenne}}{\text{écart-type}}$$
Chaque variable a alors une moyenne de 0 et un écart-type de 1.

**Pourquoi ?** La descente de gradient fait des pas de même taille pour tous les poids. Si une variable est
1000 fois plus grande qu'une autre, son poids reçoit des gradients 1000 fois plus grands : avec un seul learning
rate, impossible de bien régler les deux. Après normalisation, toutes les variables « pèsent » pareil.

**Attention :** la moyenne et l'écart-type sont calculés sur le **train uniquement**. Sinon, des informations
de la validation et du test fuiraient dans le modèle.

On divise aussi `cnt` par 1000 (on prédit des **milliers** de vélos) pour que la cible soit autour de 1 à 9
et pas autour de 4500 : les gradients restent raisonnables.

In [ ]:
mu = X[i_train].mean(axis=0)       # moyenne de chaque colonne, sur le train seulement
sigma = X[i_train].std(axis=0)     # écart-type de chaque colonne, sur le train seulement
Xn = (X - mu) / sigma              # on applique la même transformation partout
yk = y / 1000                      # cible en milliers de vélos

X_train, X_val, X_test = Xn[i_train], Xn[i_val], Xn[i_test]
y_train, y_val, y_test = yk[i_train], yk[i_val], yk[i_test]

print("moyennes des colonnes du train après normalisation :", X_train.mean(axis=0).round(2))
print("écarts-types :", X_train.std(axis=0).round(2))

### 1.6 Comment mesurer l'erreur ?

* **MSE** (erreur quadratique moyenne) $= \frac{1}{n}\sum (\hat y - y)^2$ : c'est ce qu'on **minimise** pendant
  l'entraînement, car elle est facile à dériver.
* **MAE** (erreur absolue moyenne) $= \frac{1}{n}\sum |\hat y - y|$ : c'est ce qu'on **affiche**, car elle se lit
  directement : « en moyenne, le modèle se trompe de 600 vélos par jour ».

In [ ]:
def mae(y_pred, y_true):
    # erreur absolue moyenne, reconvertie en nombre de vélos (x1000)
    return np.mean(np.abs(y_pred - y_true)) * 1000

def mse(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

resultats = {}  # on stockera ici la MAE de validation de chaque essai, pour le bilan final

---
## 2. Baseline : la prédiction la plus bête possible

**Pourquoi ce test ?** Pour avoir un point de comparaison. « Mon modèle se trompe de 600 vélos », est-ce bien ?
On ne peut le dire qu'en comparant à un modèle trivial : **prédire tous les jours la moyenne du train**,
sans regarder la météo ni rien.

In [ ]:
pred = np.full(len(y_val), y_train.mean())   # la même valeur pour tous les jours
resultats["baseline (moyenne)"] = mae(pred, y_val)
print(f"Moyenne du train : {y_train.mean() * 1000:.0f} vélos")
print(f"MAE baseline     : {resultats['baseline (moyenne)']:.0f} vélos")

**À retenir :** environ 1600 vélos d'erreur par jour. Tout modèle qui ne fait pas nettement mieux ne sert à rien.

---
## 3. Un neurone, une seule variable : la descente de gradient à la main

**Pourquoi ce test ?** C'est le cas le plus simple possible : une entrée (`temp`), un poids, un biais.
On peut tout suivre à la main et même tout dessiner.

### Le modèle

$$\hat y = w \cdot x + b$$

C'est un **neurone sans fonction d'activation** : il multiplie l'entrée par un poids et ajoute un biais.
Graphiquement c'est une **droite** : $w$ est la pente, $b$ la hauteur à l'origine.
Apprendre = trouver la droite qui colle le mieux au nuage de points.

### Comment savoir dans quel sens bouger $w$ et $b$ ?

Image classique : on est perdu dans la montagne, dans le brouillard, et on veut descendre dans la vallée.
On ne voit pas loin, mais on sent la **pente sous ses pieds**. On fait un petit pas dans le sens qui descend,
puis on recommence. C'est exactement la **descente de gradient** :

* la « montagne », c'est la perte $L(w, b)$ ;
* la « pente sous les pieds », c'est le **gradient** $\frac{\partial L}{\partial w}$, $\frac{\partial L}{\partial b}$ ;
* la taille du pas, c'est le **learning rate** $\eta$.

### Les formules

Perte : $L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$

En dérivant (règle de dérivation d'un carré) :
$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (\hat y_i - y_i)\,x_i \qquad\qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (\hat y_i - y_i)$$

Mise à jour (on va **contre** la pente, d'où le signe moins) :
$$w \leftarrow w - \eta\,\frac{\partial L}{\partial w} \qquad\qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

Intuition : si le modèle prédit trop haut ($\hat y - y > 0$) là où $x$ est grand, le gradient de $w$ est positif,
donc on **diminue** $w$. Logique.

In [ ]:
x = X_train[:, 3]      # la colonne temp (normalisée)
w, b = 0.0, 0.0        # on part de poids nuls : la droite est plate, à hauteur 0
lr = 0.1               # learning rate
pertes = []            # pour tracer l'évolution de la perte
historique_w = [w]     # pour dessiner le chemin suivi

for epoch in range(50):
    # 1) prédiction avec les poids actuels
    y_pred = w * x + b
    # 2) erreur sur chaque jour
    erreur = y_pred - y_train
    # 3) gradients (formules ci-dessus)
    grad_w = 2 * np.mean(erreur * x)
    grad_b = 2 * np.mean(erreur)
    # 4) mise à jour : un petit pas contre la pente
    w = w - lr * grad_w
    b = b - lr * grad_b
    pertes.append(mse(y_pred, y_train))
    historique_w.append(w)

print(f"Poids appris : w = {w:.3f}, b = {b:.3f}")
resultats["1 neurone, temp seule"] = mae(w * X_val[:, 3] + b, y_val)
print(f"MAE validation : {resultats['1 neurone, temp seule']:.0f} vélos")

**Comment lire $w$ et $b$ ?** $b \approx 4.5$ : un jour de température moyenne ($x = 0$), on prédit environ
4500 vélos. $w \approx 1.2$ : chaque écart-type de température en plus ajoute environ 1200 vélos.

### Ce qu'il s'est passé, en images

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) la perte au fil des époques
axes[0].plot(pertes)
axes[0].set(xlabel="époque", ylabel="MSE", title="(a) La perte diminue à chaque époque")

# (b) la "montagne" : la perte en fonction de w (b fixé à sa valeur finale), et le chemin parcouru
ws = np.linspace(-0.5, 2.5, 100)
axes[1].plot(ws, [mse(wi * x + b, y_train) for wi in ws], color="gray", label="perte L(w)")
hw = np.array(historique_w[:15])
axes[1].plot(hw, [mse(wi * x + b, y_train) for wi in hw], "o-", color="red", ms=4, label="pas de la descente")
axes[1].set(xlabel="w", ylabel="MSE", title="(b) On descend la courbe de la perte")
axes[1].legend()

# (c) la droite finale sur le nuage de points
axes[2].scatter(x, y_train, s=8, alpha=0.5)
xs = np.linspace(x.min(), x.max(), 10)
axes[2].plot(xs, w * xs + b, color="red", label="droite apprise")
axes[2].set(xlabel="temp (normalisée)", ylabel="cnt (milliers)", title="(c) La droite trouvée")
axes[2].legend()
plt.tight_layout()
plt.show()

* **(a)** La perte chute vite au début puis se stabilise : on est arrivé au fond de la vallée.
* **(b)** Chaque point rouge est une étape. Les pas sont **grands au début** (la pente est forte) et
  **de plus en plus petits** à l'approche du minimum (la pente devient presque nulle). On n'a pas besoin de
  réduire $\eta$ : c'est le gradient lui-même qui rapetisse.
* **(c)** La droite suit la tendance générale du nuage.

### Vérification

La régression linéaire a une **solution exacte** (méthode des moindres carrés, `np.polyfit`).
Si notre descente de gradient est correcte, on doit retrouver les mêmes valeurs.

In [ ]:
w_exact, b_exact = np.polyfit(x, y_train, 1)
print(f"descente de gradient : w = {w:.3f}, b = {b:.3f}")
print(f"solution exacte      : w = {w_exact:.3f}, b = {b_exact:.3f}")

**À retenir :** identiques. La descente de gradient trouve bien le minimum. Ici on aurait pu calculer la solution
directement, mais pour un réseau de neurones il n'existe **pas** de formule exacte : la descente de gradient
devient la seule option. D'où son importance.

Avec une seule variable, l'erreur reste élevée (environ 1260 vélos) : la température ne suffit pas.

---
## 4. L'effet du learning rate

**Pourquoi ce test ?** Le learning rate $\eta$ est **le réglage le plus important** de la descente de gradient.
On reprend exactement le même code et on ne change que $\eta$.

Ce qu'on s'attend à voir :
* $\eta$ trop petit : des pas minuscules, on avance très lentement ;
* $\eta$ trop grand : on saute par-dessus le fond de la vallée, et on peut même remonter de l'autre côté
  de plus en plus haut (**divergence**).

In [ ]:
def descente_1_variable(lr, n_epochs=50):
    # même boucle qu'au-dessus, rangée dans une fonction pour la relancer facilement
    w, b = 0.0, 0.0
    pertes = []
    for epoch in range(n_epochs):
        y_pred = w * x + b
        erreur = y_pred - y_train
        w = w - lr * 2 * np.mean(erreur * x)
        b = b - lr * 2 * np.mean(erreur)
        pertes.append(mse(y_pred, y_train))
    return pertes

for lr in [0.001, 0.01, 0.1, 0.5, 0.9, 1.05]:
    plt.plot(descente_1_variable(lr), label=f"lr = {lr}")
plt.yscale("log")   # échelle log : sinon la courbe qui explose écrase toutes les autres
plt.xlabel("époque"); plt.ylabel("MSE (échelle log)"); plt.title("Effet du learning rate")
plt.legend()
plt.show()

**Ce qu'on observe :**
* **0.001 et 0.01** : la perte baisse, mais au bout de 50 époques on est encore loin du minimum. Trop lent.
* **0.1** : convergence en une dizaine d'époques. Bon réglage.
* **0.5** : convergence en **une seule** étape ! Cas particulier : avec une variable normalisée, la « courbure »
  de la MSE vaut exactement 2, et $\eta = 1/2$ est alors le pas parfait qui tombe pile au fond.
* **0.9** : chaque pas dépasse le minimum et on oscille d'un côté à l'autre de la vallée, mais en se rapprochant.
* **1.05** : chaque pas dépasse **de plus en plus** : la perte explose. C'est la divergence.
  Ici la limite théorique est $\eta = 1$.

**À retenir :** il existe une plage de bons learning rates. En pratique on teste des puissances de 10
(0.1, 0.01, 0.001…) et on regarde les courbes de perte.

---
## 5. Un neurone, toutes les variables (perceptron linéaire)

**Pourquoi ce test ?** La température seule ne suffit pas. On donne maintenant au neurone les 14 variables.

### Le modèle

Même idée qu'avant, mais avec un poids par variable :
$$\hat y = w_1 x_1 + w_2 x_2 + \dots + w_{14} x_{14} + b = X w + b$$

Écrire $Xw$ (produit matrice × vecteur) calcule d'un coup la prédiction pour **tous les jours**.

Le gradient se généralise naturellement :
$$\nabla_w L = \frac{2}{n} X^\top (\hat y - y)$$
($X^\top$ fait, pour chaque variable, la somme « erreur × valeur de la variable », comme $\sum (\hat y - y)\,x$ avant.)

La fonction ci-dessous accepte aussi un paramètre `batch_size`, qui servira à la section 6.
Pour l'instant on utilise **tout le train** à chaque mise à jour.

In [ ]:
def entrainer_lineaire(X_tr, y_tr, X_va, y_va, lr=0.05, n_epochs=200, batch_size=None):
    n, d = X_tr.shape                 # n jours, d variables
    w, b = np.zeros(d), 0.0           # un poids par variable, tous à 0 au départ
    if batch_size is None:
        batch_size = n                # par défaut : tout le train d'un coup
    hist_train, hist_val = [], []
    for epoch in range(n_epochs):
        ordre = np.random.permutation(n)                  # on mélange les jours à chaque époque
        for debut in range(0, n, batch_size):             # on découpe en paquets (batchs)
            j = ordre[debut:debut + batch_size]           # indices des jours de ce batch
            erreur = X_tr[j] @ w + b - y_tr[j]            # prédiction - vérité
            w -= lr * 2 * X_tr[j].T @ erreur / len(j)     # gradient de w, puis mise à jour
            b -= lr * 2 * np.mean(erreur)                 # gradient de b, puis mise à jour
        # à la fin de chaque époque, on mesure l'erreur sur train et validation
        hist_train.append(mae(X_tr @ w + b, y_tr))
        hist_val.append(mae(X_va @ w + b, y_va))
    return w, b, hist_train, hist_val

w, b, h_tr, h_va = entrainer_lineaire(X_train, y_train, X_val, y_val)

plt.plot(h_tr, label="train"); plt.plot(h_va, label="validation")
plt.ylim(0, 2000); plt.xlabel("époque"); plt.ylabel("MAE (vélos)"); plt.legend()
plt.title("Perceptron linéaire, 14 variables")
plt.show()

resultats["perceptron linéaire"] = h_va[-1]
print(f"MAE validation : {h_va[-1]:.0f} vélos")

**Ce qu'on observe :** l'erreur passe d'environ 1260 (temp seule) à environ 630 vélos. Les deux courbes
(train et validation) sont **proches** : le modèle ne sur-apprend pas. C'est normal, il n'a que 15 paramètres
pour 439 exemples.

### Lire les poids

Grand avantage du modèle linéaire : on peut **l'interpréter**. Comme toutes les variables sont normalisées,
un poids élevé (en valeur absolue) = variable importante.

In [ ]:
ordre = np.argsort(w)    # tri des poids du plus négatif au plus positif
plt.figure(figsize=(7, 5))
plt.barh(np.array(col_names)[ordre], w[ordre] * 1000)
plt.xlabel("effet sur cnt (vélos, pour +1 écart-type)"); plt.title("Poids appris par le perceptron")
plt.show()

**Lecture :**
* `yr` est le plus grand poids positif : il y a beaucoup plus de locations en 2012 qu'en 2011 (le service grandit).
* `temp` / `atemp` : plus il fait chaud, plus on loue.
* `hum`, `windspeed`, `meteo_pluie` : humidité, vent et pluie font baisser les locations.
* `printemps` est négatif : à température égale, il y a moins de locations au printemps (début 2011 surtout).

**Piège :** `temp` et `atemp` sont presque la même information. Le modèle répartit l'effet entre les deux
de manière un peu arbitraire. Il ne faut pas sur-interpréter la valeur de chacun séparément.

**À retenir :** un seul neurone linéaire divise déjà l'erreur par 2.5 par rapport à la baseline, et reste lisible.

---
## 6. Batch, mini-batch ou SGD ?

**Pourquoi ce test ?** Jusqu'ici, pour faire **un** pas, on calculait le gradient sur **les 439 jours**. C'est
précis, mais on ne fait qu'un pas par époque. Autre option : calculer le gradient sur une partie des données
seulement, et faire plus de pas.

| Variante | Exemples par pas | Pas par époque | Gradient |
|---|---|---|---|
| **batch complet** | 439 | 1 | exact, mais peu de pas |
| **mini-batch 32** | 32 | ≈ 14 | approximatif, bon compromis |
| **SGD** (stochastique) | 1 | 439 | très bruité, mais beaucoup de pas |

On compare les trois avec le **même** learning rate (0.01) et le même nombre d'époques.

In [ ]:
for bs, nom in [(None, "batch complet"), (32, "mini-batch 32"), (1, "SGD (1 exemple)")]:
    np.random.seed(0)   # même hasard pour les trois, pour une comparaison équitable
    _, _, _, h_va = entrainer_lineaire(X_train, y_train, X_val, y_val, lr=0.01, n_epochs=50, batch_size=bs)
    plt.plot(h_va, label=nom)
plt.ylim(400, 4000); plt.xlabel("époque"); plt.ylabel("MAE validation (vélos)")
plt.title("Taille de batch (même lr = 0.01)"); plt.legend()
plt.show()

**Ce qu'on observe :**
* **batch complet** (bleu) : courbe lisse mais très lente. En 50 époques, seulement 50 pas : on est loin du but.
* **SGD** (vert) : arrive au minimum dès la première époque (439 pas !), mais la courbe **tremble** : chaque pas
  est calculé sur un seul jour, donc il part parfois dans une mauvaise direction.
* **mini-batch 32** (orange) : converge en une dizaine d'époques avec une courbe lisse.

**À retenir :** le mini-batch combine les avantages des deux. C'est le choix par défaut (32 dans Keras).
Autre avantage non visible ici : sur GPU, calculer 32 exemples coûte à peu près le même temps qu'un seul.

---
## 7. Le perceptron historique de Rosenblatt (classification)

**Pourquoi ce test ?** Le « perceptron » au sens strict (Rosenblatt, 1958) est l'ancêtre des réseaux de neurones.
Il ne fait pas de régression mais de la **classification** : il répond oui ou non.

On transforme donc notre problème : **ce jour est-il « chargé » ?** (plus de vélos que la médiane → classe 1,
sinon classe 0). Avec la médiane, on a exactement 50 % de chaque classe.

### Le modèle

$$\hat y = \begin{cases} 1 & \text{si } x \cdot w + b > 0 \\ 0 & \text{sinon} \end{cases}$$

Le neurone calcule toujours sa somme pondérée, mais applique un **seuil** : c'est sa fonction d'activation.

### La règle d'apprentissage (pas de gradient ici !)

On passe les jours un par un. Pour chaque jour :
$$w \leftarrow w + \eta\,(y - \hat y)\,x \qquad b \leftarrow b + \eta\,(y - \hat y)$$

* bonne réponse : $y - \hat y = 0$, **on ne change rien** ;
* il a dit 0 au lieu de 1 : $y - \hat y = +1$, on **ajoute** $x$ à $w$ (le score de ce jour augmentera) ;
* il a dit 1 au lieu de 0 : $y - \hat y = -1$, on **retire** $x$.

In [ ]:
seuil = np.median(y_train)
c_train = (y_train > seuil).astype(int)   # 1 = jour chargé, 0 = jour calme
c_val = (y_val > seuil).astype(int)
print(f"seuil = {seuil * 1000:.0f} vélos | proportion de jours chargés dans le train : {c_train.mean():.0%}")

w, b, lr = np.zeros(X_train.shape[1]), 0.0, 0.1
acc_train, acc_val = [], []
for epoch in range(30):
    for i in np.random.permutation(len(X_train)):        # un jour à la fois, dans un ordre aléatoire
        pred = int(X_train[i] @ w + b > 0)               # réponse du perceptron : 0 ou 1
        w += lr * (c_train[i] - pred) * X_train[i]       # correction seulement s'il s'est trompé
        b += lr * (c_train[i] - pred)
    # accuracy = proportion de bonnes réponses
    acc_train.append(np.mean((X_train @ w + b > 0) == c_train))
    acc_val.append(np.mean((X_val @ w + b > 0) == c_val))

plt.plot(acc_train, label="train"); plt.plot(acc_val, label="validation")
plt.xlabel("époque"); plt.ylabel("accuracy"); plt.title("Perceptron de Rosenblatt"); plt.legend()
plt.show()
print(f"accuracy validation finale : {acc_val[-1]:.0%}")

**Ce qu'on observe :** entre 75 et 90 % de bonnes réponses selon l'époque, et la courbe **ne se stabilise jamais**.

**Pourquoi ?** Le perceptron sépare l'espace par un plan (une « droite » en dimension 14). Si un tel plan peut
séparer parfaitement les deux classes, le perceptron le trouve (c'est un théorème). Mais ici il y a des jours
limites (un jour à 4400 vélos ressemble beaucoup à un jour à 4600) : **aucun plan n'est parfait**. Le perceptron
continue alors de corriger à chaque erreur, et chaque correction en crée d'autres : il oscille sans fin.

### La solution : une activation dérivable

On remplace le seuil (marche d'escalier, non dérivable) par une **sigmoïde** :
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
qui passe en douceur de 0 à 1. La sortie devient une **probabilité** (« 80 % de chances que le jour soit chargé »).
On peut alors utiliser la descente de gradient sur une perte continue, l'**entropie croisée**. C'est la
**régression logistique**. Son gradient a une forme très simple : $\nabla_w = \frac{1}{n} X^\top (p - y)$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w, b, lr = np.zeros(X_train.shape[1]), 0.0, 0.5
acc_val_sig = []
for epoch in range(30):
    p = sigmoid(X_train @ w + b)                           # probabilité d'être un jour chargé
    w -= lr * X_train.T @ (p - c_train) / len(c_train)     # gradient de l'entropie croisée
    b -= lr * np.mean(p - c_train)
    acc_val_sig.append(np.mean((sigmoid(X_val @ w + b) > 0.5) == c_val))   # > 50 % -> classe 1

plt.plot(acc_val, label="Rosenblatt (seuil)")
plt.plot(acc_val_sig, label="neurone sigmoïde (descente de gradient)")
plt.xlabel("époque"); plt.ylabel("accuracy validation"); plt.legend()
plt.show()

**À retenir :** avec la sigmoïde, la courbe monte **régulièrement et se stabilise**. Au lieu de sursauter à
chaque erreur, le modèle minimise une perte globale et s'arrête au meilleur compromis. C'est pour ça que les
réseaux modernes utilisent des activations dérivables et la descente de gradient.

---
## 8. Un réseau à une couche cachée (MLP)

**Pourquoi ce test ?** Un neurone linéaire ne peut faire que des relations **en ligne droite**. Or on a vu à la
section 1.3 que l'effet de la température n'est pas tout à fait linéaire (ça plafonne quand il fait très chaud).
Pour capturer ce genre d'effet, on ajoute des neurones **entre** l'entrée et la sortie : une **couche cachée**.

### Le modèle

```
14 variables  ──►  [ 16 neurones cachés, activation ReLU ]  ──►  1 neurone de sortie  ──►  ŷ
```

$$h = \text{ReLU}(X W_1 + b_1) \qquad\qquad \hat y = h\,W_2 + b_2$$

* $W_1$ : matrice 14 × 16 (chaque neurone caché a 14 poids), $b_1$ : 16 biais ;
* $W_2$ : 16 × 1 (le neurone de sortie combine les 16 neurones cachés), $b_2$ : 1 biais.

**ReLU** $(z) = \max(0, z)$ : on garde $z$ s'il est positif, sinon 0. C'est ce « coude » qui rend le réseau
**non linéaire**. Sans lui, $h W_2$ serait juste une combinaison linéaire de combinaisons linéaires…
donc encore un modèle linéaire, et la couche cachée ne servirait à rien.

### La rétropropagation (backpropagation)

On veut le gradient de la perte pour **chaque** poids, y compris ceux de la couche cachée. On utilise la
**règle de dérivation en chaîne** : on part de l'erreur à la sortie et on la fait « remonter » couche par couche.

1. Erreur en sortie : $e = \frac{2}{n}(\hat y - y)$
2. Gradient des poids de sortie : $\nabla W_2 = h^\top e$ (comme pour le perceptron linéaire, avec $h$ comme entrée)
3. On renvoie l'erreur vers la couche cachée : $\delta = (e\,W_2^\top) \odot \mathbb{1}[h > 0]$
   * $e\,W_2^\top$ : chaque neurone caché reçoit une part de l'erreur, proportionnelle à son poids vers la sortie ;
   * $\mathbb{1}[h > 0]$ : dérivée de ReLU (1 si le neurone était actif, 0 sinon) : un neurone éteint ne transmet rien.
4. Gradient des poids cachés : $\nabla W_1 = X^\top \delta$

### Le code

On découpe en petites fonctions pour que chaque étape soit lisible. La fonction `entrainer_mlp` contient
aussi trois façons de mettre à jour les poids (`sgd`, `momentum`, `adam`) et une option `l2`,
expliquées aux sections 10 et 11. Pour l'instant on utilise `sgd`.

In [ ]:
def initialiser(n_entrees, n_caches, rng):
    # Poids de départ aléatoires et petits (s'ils étaient tous à 0, tous les neurones cachés
    # calculeraient la même chose et apprendraient la même chose : ils resteraient identiques).
    # Échelle sqrt(2 / n_entrees) : initialisation "de He", adaptée à ReLU.
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_entrees), (n_entrees, n_caches)),
        "b1": np.zeros(n_caches),
        "W2": rng.normal(0, np.sqrt(1 / n_caches), (n_caches, 1)),
        "b2": np.array([y_train.mean()]),   # astuce : la sortie part de la moyenne (= la baseline)
    }


def propagation_avant(params, X):
    # entrée -> couche cachée (ReLU) -> sortie
    h = np.maximum(0, X @ params["W1"] + params["b1"])
    y_pred = (h @ params["W2"] + params["b2"]).ravel()
    return y_pred, h


def retropropagation(params, X, y, y_pred, h, l2=0.0):
    # gradients de la MSE par rapport à chaque paramètre (étapes 1 à 4 ci-dessus)
    e = (2 * (y_pred - y) / len(y)).reshape(-1, 1)          # 1) erreur en sortie
    grads = {
        "W2": h.T @ e + l2 * params["W2"],                  # 2) gradient des poids de sortie
        "b2": e.sum(axis=0),
    }
    delta = (e @ params["W2"].T) * (h > 0)                  # 3) erreur renvoyée à la couche cachée
    grads["W1"] = X.T @ delta + l2 * params["W1"]           # 4) gradient des poids cachés
    grads["b1"] = delta.sum(axis=0)
    return grads

In [ ]:
def entrainer_mlp(n_caches=16, lr=0.01, n_epochs=300, batch_size=32, optim="sgd", l2=0.0, seed=0):
    rng = np.random.default_rng(seed)
    params = initialiser(X_train.shape[1], n_caches, rng)
    # mémoires utilisées par momentum et Adam (section 10)
    vitesse = {k: np.zeros_like(v) for k, v in params.items()}
    carre = {k: np.zeros_like(v) for k, v in params.items()}
    t = 0   # nombre de pas effectués (utile pour Adam)

    hist_train, hist_val = [], []
    for epoch in range(n_epochs):
        ordre = rng.permutation(len(X_train))
        for debut in range(0, len(X_train), batch_size):
            j = ordre[debut:debut + batch_size]
            # 1) prédire  2) calculer les gradients
            y_pred, h = propagation_avant(params, X_train[j])
            grads = retropropagation(params, X_train[j], y_train[j], y_pred, h, l2)
            # 3) mettre à jour chaque paramètre
            t += 1
            for k in params:
                if optim == "sgd":           # descente de gradient classique
                    params[k] -= lr * grads[k]
                elif optim == "momentum":    # on garde 90 % du pas précédent (inertie)
                    vitesse[k] = 0.9 * vitesse[k] - lr * grads[k]
                    params[k] += vitesse[k]
                elif optim == "adam":        # momentum + pas adapté à chaque poids
                    vitesse[k] = 0.9 * vitesse[k] + 0.1 * grads[k]
                    carre[k] = 0.999 * carre[k] + 0.001 * grads[k] ** 2
                    m_hat = vitesse[k] / (1 - 0.9 ** t)       # correction du biais de départ
                    v_hat = carre[k] / (1 - 0.999 ** t)
                    params[k] -= lr * m_hat / (np.sqrt(v_hat) + 1e-8)
        hist_train.append(mae(propagation_avant(params, X_train)[0], y_train))
        hist_val.append(mae(propagation_avant(params, X_val)[0], y_val))

    predire = lambda X: propagation_avant(params, X)[0]
    return params, predire, hist_train, hist_val

**Avant d'entraîner : vérifier la rétropropagation.** Une erreur de signe ou d'indice dans les gradients ne
fait pas planter le code, le réseau apprend juste mal. On vérifie donc en comparant avec une dérivée
**numérique** : on bouge un poids d'un tout petit $\varepsilon$ et on regarde de combien la perte change :
$$\frac{\partial L}{\partial w} \approx \frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$$

In [ ]:
p_test = initialiser(X_train.shape[1], 4, np.random.default_rng(1))
Xp, yp = X_train[:20], y_train[:20]
y_pred, h = propagation_avant(p_test, Xp)
g = retropropagation(p_test, Xp, yp, y_pred, h)

eps = 1e-5
p_test["W1"][0, 0] += eps;     L_plus = mse(propagation_avant(p_test, Xp)[0], yp)
p_test["W1"][0, 0] -= 2 * eps; L_moins = mse(propagation_avant(p_test, Xp)[0], yp)
p_test["W1"][0, 0] += eps
print(f"gradient par rétropropagation : {g['W1'][0, 0]:.8f}")
print(f"gradient numérique            : {(L_plus - L_moins) / (2 * eps):.8f}")

Les deux valeurs sont identiques : notre rétropropagation est juste. On peut entraîner.

In [ ]:
params, predire, h_tr, h_va = entrainer_mlp(n_caches=16, lr=0.01, n_epochs=300)

plt.plot(h_tr, label="train"); plt.plot(h_va, label="validation")
plt.ylim(0, 1500); plt.xlabel("époque"); plt.ylabel("MAE (vélos)")
plt.title("MLP, 16 neurones cachés, SGD"); plt.legend()
plt.show()

resultats["MLP 16 neurones, SGD, 300 époques"] = h_va[-1]
print(f"MAE validation : minimum {min(h_va):.0f} (époque {np.argmin(h_va)}), finale {h_va[-1]:.0f}")

**Ce qu'on observe :**
* le MLP fait mieux que le perceptron linéaire (environ 545 contre 630 vélos) grâce à la non-linéarité ;
* mais regardez l'**écart** entre les deux courbes : la MAE de train continue de baisser (≈ 400) alors que celle
  de validation stagne. Le réseau commence à apprendre des détails propres au train qui ne se généralisent pas :
  c'est le début du **sur-apprentissage**.

**Comment lire des courbes d'apprentissage** (outil principal du chapitre 2) :

| Ce qu'on voit | Diagnostic | Que faire ? |
|---|---|---|
| train et validation hautes et proches | **sous-apprentissage** : modèle trop simple | plus de neurones / couches, entraîner plus longtemps |
| train basse, validation nettement plus haute (ou qui remonte) | **sur-apprentissage** | régularisation, moins de neurones, plus de données, early stopping |
| train et validation basses et proches | bon compromis | 👍 |

---
## 9. Combien de neurones cachés ?

**Pourquoi ce test ?** Le nombre de neurones règle la **capacité** du réseau : sa capacité à représenter des
relations compliquées. Trop peu → il ne peut pas capturer les effets non linéaires. Trop → il a assez de
paramètres pour apprendre le bruit du train par cœur.

Nombre de paramètres pour $H$ neurones cachés : $14H + H + H + 1 = 16H + 1$. Avec 256 neurones, cela fait
**4097 paramètres pour 439 jours d'entraînement**.

In [ ]:
print(f"{'neurones':>8} | {'paramètres':>10} | {'MAE train':>9} | {'MAE validation':>14}")
for n in [1, 4, 16, 64, 256]:
    _, _, h_tr, h_va = entrainer_mlp(n_caches=n, lr=0.01, n_epochs=300)
    print(f"{n:8d} | {16 * n + 1:10d} | {h_tr[-1]:9.0f} | {h_va[-1]:9.0f} (min {min(h_va):.0f})")

**Ce qu'on observe :**
* **1 neurone** : à peine mieux que le perceptron linéaire. Normal : un seul neurone caché ne peut faire qu'un « coude ».
* de **4 à 64** neurones : la validation s'améliore, 64 est le meilleur ici.
* **256** neurones : la MAE de train est la plus basse (≈ 290) mais la validation **remonte** (≈ 550).
  Le réseau mémorise au lieu de comprendre.

**À retenir :** plus de capacité ≠ meilleur modèle. Avec peu de données (439 jours), un réseau trop gros
sur-apprend. Toujours juger sur la **validation**, jamais sur le train.

---
## 10. Les optimiseurs : SGD, momentum, Adam

**Pourquoi ce test ?** La descente de gradient « simple » (SGD) a deux défauts :
* elle est lente quand la vallée est longue et plate (petits gradients = petits pas) ;
* le même learning rate est utilisé pour tous les poids, alors que certains auraient besoin de grands pas
  et d'autres de petits.

Les optimiseurs modernes modifient **la façon de faire le pas** :

| Optimiseur | Idée | Image |
|---|---|---|
| **SGD** | $\theta \leftarrow \theta - \eta\,g$ | un marcheur qui suit la pente |
| **Momentum** | on ajoute 90 % du pas précédent : $v \leftarrow 0.9\,v - \eta\,g$, puis $\theta \leftarrow \theta + v$ | une bille qui prend de l'**élan** : elle accélère dans les longues descentes et passe les petites bosses |
| **Adam** | momentum + chaque poids a son propre pas, divisé par la taille moyenne de ses gradients récents | une bille qui **adapte sa vitesse** au terrain pour chaque direction |

Adam est utilisé avec un learning rate plus petit (0.001, sa valeur par défaut dans Keras).

In [ ]:
for optim, lr in [("sgd", 0.01), ("momentum", 0.01), ("adam", 0.001)]:
    _, _, _, h_va = entrainer_mlp(n_caches=16, lr=lr, n_epochs=150, optim=optim)
    plt.plot(h_va, label=f"{optim} (lr={lr})")
    resultats[f"MLP 16, {optim}, 150 époques"] = h_va[-1]
plt.ylim(400, 1500); plt.xlabel("époque"); plt.ylabel("MAE validation (vélos)")
plt.title("Comparaison des optimiseurs"); plt.legend()
plt.show()

**Ce qu'on observe :**
* **Momentum** est le plus rapide : il atteint son meilleur niveau en une dizaine d'époques grâce à son élan.
  Mais ensuite sa courbe devient bruitée et remonte : il sur-apprend plus tôt (l'élan le fait continuer à « creuser » le train).
* **SGD** descend plus lentement mais régulièrement.
* **Adam** démarre plus lentement (petit learning rate) mais finit **le plus bas**.

**À retenir :** l'optimiseur change surtout la **vitesse** et la **stabilité** de l'apprentissage.
Adam est le choix par défaut le plus sûr, car il marche bien sans trop régler le learning rate.

---
## 11. Lutter contre le sur-apprentissage : L2 et early stopping

**Pourquoi ce test ?** On a vu que les réseaux un peu gros sur-apprennent. Deux remèdes simples :

### La régularisation L2

On ajoute à la perte une **pénalité** sur la taille des poids :
$$L_{\text{total}} = \text{MSE} + \frac{\lambda}{2}\sum w^2$$
Le modèle doit maintenant faire un compromis : bien prédire **et** garder des poids petits. Des poids petits
donnent une fonction plus « lisse », moins capable de suivre chaque point bruité.

Dans le gradient, ça ajoute simplement $+\lambda w$ (c'est le `+ l2 * params["W1"]` dans `retropropagation`).
À chaque pas, chaque poids est donc légèrement tiré vers 0 (on parle aussi de *weight decay*).

$\lambda$ règle la force : 0 = pas de régularisation, trop grand = les poids sont écrasés et le modèle
devient trop simple (sous-apprentissage).

On teste sur un MLP de 64 neurones avec momentum (une configuration qui sur-apprend nettement).

In [ ]:
print(f"{'λ':>6} | {'MAE train':>9} | {'MAE validation':>14} | écart validation - train")
for l2 in [0, 0.001, 0.01, 0.1]:
    _, _, h_tr, h_va = entrainer_mlp(n_caches=64, lr=0.01, n_epochs=300, optim="momentum", l2=l2)
    plt.plot(h_va, label=f"λ = {l2}")
    print(f"{l2:6} | {h_tr[-1]:9.0f} | {h_va[-1]:14.0f} | {h_va[-1] - h_tr[-1]:5.0f}")
plt.ylim(400, 900); plt.xlabel("époque"); plt.ylabel("MAE validation (vélos)")
plt.title("MLP 64 neurones : effet de la pénalité L2"); plt.legend()
plt.show()

**Ce qu'on observe :**
* $\lambda = 0$ et $0.001$ : grand écart train/validation (≈ 300 vélos) : sur-apprentissage.
* $\lambda = 0.01$ : l'écart diminue nettement et la validation est la meilleure.
* $\lambda = 0.1$ : l'écart est presque nul, mais le train lui-même devient moins bon (≈ 485) : on commence
  à **trop** contraindre le modèle.

### L'early stopping (arrêt précoce)

Idée très simple : puisque la validation finit par stagner ou remonter, on **s'arrête à la meilleure époque**.
On repère cette époque, puis on réentraîne le modèle (même graine aléatoire) en s'arrêtant là.
Keras le fait automatiquement avec le callback `EarlyStopping(restore_best_weights=True)`.

In [ ]:
_, _, _, h_va = entrainer_mlp(n_caches=64, lr=0.01, n_epochs=300, optim="momentum", l2=0.01)
meilleure_epoque = int(np.argmin(h_va)) + 1
print(f"meilleure époque : {meilleure_epoque} (MAE validation {min(h_va):.0f})")

# on réentraîne en s'arrêtant à la meilleure époque : c'est notre modèle final
params, predire_final, _, h_va = entrainer_mlp(n_caches=64, lr=0.01, n_epochs=meilleure_epoque,
                                               optim="momentum", l2=0.01)
resultats["MLP 64 + L2 + early stopping"] = h_va[-1]

**À retenir :** la régularisation ne rend pas le modèle meilleur sur le train, elle le rend meilleur sur des
**données qu'il n'a jamais vues**, ce qui est le seul objectif qui compte.

---
## 12. Bilan et évaluation finale

On classe tous les essais selon leur MAE de **validation** et on retient le meilleur.

In [ ]:
print(f"{'modèle':37s} MAE validation")
print("-" * 54)
for nom, v in sorted(resultats.items(), key=lambda kv: kv[1]):
    print(f"{nom:37s} {v:8.0f} vélos")

meilleur = min(resultats, key=resultats.get)
print(f"\nModèle retenu : {meilleur}")

### Évaluation sur le test

Maintenant, et **seulement maintenant**, on regarde le jeu de test, une seule fois. Ce chiffre est la vraie
estimation de l'erreur du modèle sur des jours futurs.

In [ ]:
y_pred_test = predire_final(X_test)
print(f"MAE test : {mae(y_pred_test, y_test):.0f} vélos")

plt.figure(figsize=(5, 5))
plt.scatter(y_test * 1000, y_pred_test * 1000, s=10)
plt.plot([0, 9000], [0, 9000], "r--", label="prédiction parfaite")
plt.xlabel("vrai cnt"); plt.ylabel("cnt prédit"); plt.title("Test : prédiction vs réalité")
plt.legend()
plt.show()

**Lecture :** chaque point est un jour du test. Plus il est proche de la diagonale rouge, meilleure est la
prédiction. Les points sont bien alignés ; les plus gros écarts concernent quelques jours atypiques (jours de
tempête, jours fériés particuliers…) que le modèle ne peut pas deviner avec ces variables.

La MAE de test est un peu plus élevée que celle de validation : c'est normal, on avait choisi le modèle
**parce qu'il** était bon sur la validation, il y a donc un léger optimisme dans ce chiffre-là.

---
## Ce qu'il faut retenir

1. **Un neurone** = une somme pondérée des entrées (+ éventuellement une activation). Sans activation,
   c'est une régression linéaire.
2. **La descente de gradient** : calculer la pente de la perte, faire un petit pas dans le sens opposé,
   recommencer. On a vérifié qu'elle retrouve la solution exacte.
3. **Le learning rate** est le réglage le plus sensible : trop petit = lent, trop grand = divergence.
4. **Le mini-batch** est le bon compromis entre précision du gradient et nombre de mises à jour.
5. **Le perceptron de Rosenblatt** oscille quand les classes ne sont pas parfaitement séparables ;
   une activation dérivable (sigmoïde) + la descente de gradient règlent le problème.
6. **Une couche cachée avec ReLU** permet des relations non linéaires ; la **rétropropagation** calcule
   tous les gradients en faisant remonter l'erreur.
7. **Plus de neurones** = plus de capacité, mais aussi plus de risque de **sur-apprentissage** ;
   on le détecte avec les courbes train/validation.
8. **Momentum et Adam** accélèrent l'apprentissage ; Adam est le choix par défaut.
9. **L2 et early stopping** limitent le sur-apprentissage.

| Modèle | MAE (vélos/jour) |
|---|---|
| moyenne | ≈ 1600 |
| 1 neurone, température seule | ≈ 1260 |
| perceptron linéaire, 14 variables | ≈ 630 |
| meilleur MLP (64 neurones + L2 + early stopping) | ≈ 470 en validation, ≈ 500 en test |

## Pour aller plus loin (à essayer soi-même)

* Section 4 : tester `lr = 0.99` puis `lr = 1.01` : on voit la frontière de la divergence.
* Section 5 : retirer `atemp` et vérifier que le poids de `temp` absorbe son effet.
* Section 8 : remplacer ReLU par une activation linéaire ($h = XW_1 + b_1$) : le MLP ne fait plus mieux que le perceptron linéaire. Pourquoi ?
* Section 9 : ajouter une deuxième couche cachée.
* Ajouter `mnth` et `weekday` en one-hot (ou en encodage cyclique sin/cos) et voir si l'erreur baisse.